# Corrected Poke-In/Poke-Out Extraction + InSeq vs. OutSeq Spectrogram EDA (Mitt, Buchanan)

**Simple version:** we're fixing a bug where our code accidentally looked too far ahead in time and
grabbed the wrong event, then we're making a picture that shows, for every moment around the rat
sticking its nose in, which "pitches" (frequencies) of brain activity were loud or quiet, so we can SEE
the pattern instead of guessing at it.

**In-depth version:** this notebook does two things. First, it fixes a bug found in notebook 019: the
search for each trial's Poke-Out event sometimes reached into the NEXT trial's events (evidenced by
gap values up to 35+ minutes, physically impossible for a single nose-poke). This version bounds the
search strictly between the current trial's Poke-In and the next trial's Poke-In, so it can never leak
across trials. Second, it builds the spectrogram EDA requested directly by the lab: a time-frequency
plot (time on x-axis, frequency on y-axis, power as color) for InSeq vs. OutSeq trials, aligned
separately to Poke-In and to the now-corrected Poke-Out. This is meant to be looked at BEFORE deciding
on bands or window length, per the explicit guidance to gain insight from the shape of this figure
first, rather than continuing with band choices made by convention.

**Why Mitt and Buchanan specifically:** per the advisor's guidance to prioritize these two rats when
settling on a universal window (and to treat Superchris's results with more caution as a reference
case), rather than defaulting to whichever rat happens to look best.


In [17]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from pathlib import Path

from src.preprocessing import build_labels, get_sampling_rate

PRIORITY_RATS = ['080718_mitt', '090420_buchanan']
raw_dir = Path('../data/raw')
session_dirs = [p for p in sorted(raw_dir.iterdir()) if p.is_dir() and p.name in PRIORITY_RATS]
print(f"Using: {[p.name for p in session_dirs]}")


Using: ['080718_mitt', '090420_buchanan']


## Part A: Corrected Poke-In / Poke-Out Extraction

**Simple version:** for each trial, find the moment the rat's nose went IN (already know this), then
look for the moment its nose came back OUT, but only look at times that still belong to THIS trial, not
accidentally peeking into the next one.

**In-depth version:** same approach as notebook 019 (find the `PokeEvents` entry after each trial's
Poke-In marker), but now the search window is explicitly capped at the NEXT trial's own Poke-In index.
This guarantees the "Poke-Out" we find can never actually belong to a different trial.

**Why bound it this way, instead of e.g. a fixed time cutoff (like "only look within 5 seconds"):**
a fixed cutoff would need a guessed number that might be wrong for some trials (too short for a rat
that lingers, too long for a rat that's quick), and could still accidentally include a same-trial extra
event beyond that guess. Bounding by "the next trial's marker" uses information we already know for
certain (exactly where the next trial starts) rather than a number we'd have to guess, so it's both
simpler and can't be wrong in the way a guessed cutoff could be.


In [18]:
def extract_poke_in_out(bvr_data, bvr_keys, timebin):
    labels = build_labels(bvr_data, bvr_keys)
    trial_idx = labels['trial_idx']
    poke_events_row = bvr_data[bvr_keys.index('PokeEvents')]
    poke_event_idx = np.where(poke_events_row != 0)[0]

    poke_out_gaps_ms = []
    poke_out_sample_idx = []  # aligned to trial_idx, -1 if not found
    for i, t in enumerate(trial_idx):
        next_trial_bound = trial_idx[i + 1] if i + 1 < len(trial_idx) else len(timebin)
        # only look for events strictly between this trial's marker and the next trial's marker
        candidates = poke_event_idx[(poke_event_idx > t) & (poke_event_idx < next_trial_bound)]
        if len(candidates) == 0:
            poke_out_gaps_ms.append(np.nan)
            poke_out_sample_idx.append(-1)
            continue
        poke_out_idx = candidates[0]  # first event after Poke-In, bounded, is the real candidate Poke-Out
        gap_ms = (timebin[poke_out_idx] - timebin[t]) * 1000
        poke_out_gaps_ms.append(gap_ms)
        poke_out_sample_idx.append(poke_out_idx)

    return {
        'trial_idx': trial_idx,
        'inseq_outseq': labels['inseq_outseq'],
        'poke_out_gap_ms': np.array(poke_out_gaps_ms),
        'poke_out_sample_idx': np.array(poke_out_sample_idx),
    }


corrected_results = {}
for session_dir in session_dirs:
    session_name = session_dir.name
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    timebin = bvr_data[bvr_keys.index('TimeBin')]

    result = extract_poke_in_out(bvr_data, bvr_keys, timebin)
    valid_gaps = result['poke_out_gap_ms'][~np.isnan(result['poke_out_gap_ms'])]
    corrected_results[session_name] = result

    print(f"\n{session_name}")
    print(f"  Trials with a found Poke-Out: {len(valid_gaps)} of {len(result['trial_idx'])}")
    print(f"  Poke-In to Poke-Out gap: median={np.median(valid_gaps):.1f}ms, "
          f"min={np.min(valid_gaps):.1f}ms, max={np.max(valid_gaps):.1f}ms, std={np.std(valid_gaps):.1f}ms")



080718_mitt
  Trials with a found Poke-Out: 292 of 292
  Poke-In to Poke-Out gap: median=1389.5ms, min=18.0ms, max=2625.0ms, std=318.3ms

090420_buchanan
  Trials with a found Poke-Out: 270 of 270
  Poke-In to Poke-Out gap: median=1299.0ms, min=274.0ms, max=1907.0ms, std=165.5ms


**What to look for (simple):** the "how long was the nose in the port" number should now be a normal,
believable amount of time (like half a second to a couple seconds), not anything showing minutes.

**What to look for (in-depth):** the median gap should be plausible AND the max should no longer show
the absurd multi-minute values from notebook 019. If both hold, Poke-Out is confirmed real and reliably
extractable. If the max is still large, that would mean some trials genuinely have no second
`PokeEvents` entry before the next trial (worth checking how many `n_found` vs. `n_total` shows).


## Part B: Build Extended Trial Snippets for the Spectrogram

**Simple version:** for every trial, cut out a chunk of brain data starting a bit before the nose goes
in and continuing well after, wide enough that we can see both the "nose in" moment and the "nose out"
moment in the same picture.

**In-depth version:** each snippet spans -1000ms to +3000ms relative to Poke-In (wide enough to
comfortably include both Poke-In and the typical ~1.3-1.4 second Poke-Out gap found in Part A), using the
same timestamp-precise extraction established in notebook 007 (so this is correct regardless of the
non-uniform sampling rate issue found back then).

**Why -1000 to +3000ms specifically, rather than a tighter window:** the spectrogram's whole purpose is
exploratory, we don't yet know where the interesting activity is, so the window needs to be generous
enough to not accidentally cut off something important. -1000ms gives a "before anything happens"
baseline for comparison, and +3000ms comfortably covers Poke-In, the ~1.3s gap to Poke-Out, and some
time after Poke-Out too. This is intentionally wider than any of the final classification windows used
elsewhere in the project, those were narrowed down AFTER seeing evidence like this; this step comes
before that narrowing.


In [ ]:
SNIPPET_PRE_MS = 1000
SNIPPET_POST_MS = 3000
SPEC_NPERSEG = 64  # shared with Part C/D so every spectrogram uses the same time/frequency grid
BASELINE_WINDOW_MS = (-1000, -500)  # pre-Poke-In window used to baseline-correct every spectrogram

def extract_snippet(lfp_data, timebin, poke_idx, pre_ms, post_ms):
    poke_time = timebin[poke_idx]
    start_time = poke_time - pre_ms / 1000
    end_time = poke_time + post_ms / 1000
    if start_time < timebin[0] or end_time > timebin[-1]:
        return None
    start_idx = np.searchsorted(timebin, start_time)
    end_idx = np.searchsorted(timebin, end_time)
    return lfp_data[:, start_idx:end_idx]


def compute_trial_baseline_power(snip, fs, nperseg, snippet_pre_ms, baseline_window_ms):
    """
    Per-trial, per-frequency baseline power (averaged across channels), taken from the
    pre-Poke-In window of THIS snippet.

    This is the fix for the "flat, all-horizontal-bands" spectrograms: raw LFP power falls off
    steeply with frequency (roughly 1/f), so a single trial's power at 5 Hz can be 100x its power
    at 100 Hz. Averaging RAW power across trials and plotting it directly (what the notebook did
    before) means the color scale ends up set almost entirely by that frequency-to-frequency
    falloff -- any real *time-locked* change within one frequency band (the thing we actually care
    about) is a tiny fraction of that range and disappears into flat horizontal stripes. This is
    not about which event we align to; it shows up the same way whether we align to Poke-In,
    Poke-Out, or something earlier.

    Expressing power as dB relative to each trial's own pre-Poke-In baseline removes the 1/f
    falloff per frequency and makes time-locked changes visible.
    """
    n_channels = snip.shape[0]
    channel_Sxx = []
    for ch in range(n_channels):
        f, t, Sxx = signal.spectrogram(snip[ch], fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
        channel_Sxx.append(Sxx)
    trial_Sxx = np.mean(channel_Sxx, axis=0)  # (n_freqs, n_times), averaged across channels
    times_ms = (t - snippet_pre_ms / 1000) * 1000  # 0 = Poke-In, since this snip starts at Poke-In - pre_ms
    baseline_mask = (times_ms >= baseline_window_ms[0]) & (times_ms < baseline_window_ms[1])
    if not baseline_mask.any():
        raise ValueError("No spectrogram time bins fall inside BASELINE_WINDOW_MS; check the window.")
    return trial_Sxx[:, baseline_mask].mean(axis=1)  # (n_freqs,)


snippets_by_rat = {}
for session_dir in session_dirs:
    session_name = session_dir.name
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    lfp = np.load(session_dir / f'{session_name}_lfp.npz', allow_pickle=True)
    lfp_data = lfp['data']
    timebin = bvr_data[bvr_keys.index('TimeBin')]
    avg_fs = get_sampling_rate(bvr_data, bvr_keys)

    result = corrected_results[session_name]
    snippets, inseq_labels, poke_out_gaps, baseline_powers = [], [], [], []
    for i, t in enumerate(result['trial_idx']):
        snip = extract_snippet(lfp_data, timebin, t, SNIPPET_PRE_MS, SNIPPET_POST_MS)
        if snip is not None:
            snippets.append(snip)
            inseq_labels.append(result['inseq_outseq'][i])
            poke_out_gaps.append(result['poke_out_gap_ms'][i])
            baseline_powers.append(
                compute_trial_baseline_power(snip, avg_fs, SPEC_NPERSEG, SNIPPET_PRE_MS, BASELINE_WINDOW_MS)
            )

    snippets_by_rat[session_name] = {
        'snippets': snippets,  # list of (n_channels, n_samples), lengths may vary slightly, resampled below
        'inseq_labels': np.array(inseq_labels),
        'poke_out_gaps_ms': np.array(poke_out_gaps),
        # list of (n_freqs,) arrays, same order/length as 'snippets': this trial's own pre-Poke-In
        # baseline power. Reused for BOTH Part C and Part D so both alignments are normalized
        # against the exact same reference.
        'baseline_power': baseline_powers,
        'fs': avg_fs,
    }
    print(f"{session_name}: {len(snippets)} snippets extracted")


## Part C: Compute and Plot Spectrograms, InSeq vs. OutSeq, Poke-In Aligned

**Simple version:** we're making a picture that shows, at every moment in time, how strong each
"pitch" (frequency) of brain wave was, for correct-order trials and wrong-order trials separately, so we
can compare them side by side and see with our own eyes if and how they differ.

**In-depth version:** for each trial, compute a spectrogram (`scipy.signal.spectrogram`) per channel,
then average across channels and across all trials within each condition (InSeq, OutSeq). Time is NOT
collapsed here, this is exactly the "time vs. frequency" figure requested by the lab, letting the
frequency content over time speak for itself before any band or window decision is made.

**Why a spectrogram (time-resolved) instead of the single FFT-based band-power numbers used everywhere
else in this project so far:** every prior notebook collapsed each window into one number per band
(e.g. "total theta power in this 250ms chunk"), which is efficient for training a classifier but throws
away exactly the information we need right now, WHEN within the window something happens. A spectrogram
keeps that timing information visible, which is the whole point of this exploratory step: to see
whether power in a given band is constant throughout the trial (in which case collapsing time later is
fine) or changes sharply at a specific moment (in which case collapsing time would have been hiding
something important).

**Why `scipy.signal.spectrogram` specifically, rather than building it manually:** it's a standard,
well-tested implementation of the short-time Fourier transform (repeatedly computing an FFT over small,
overlapping chunks of the signal, exactly the same FFT idea used throughout this project, just applied
many times across a sliding sub-window instead of once for the whole window). Using the standard library
function avoids reinventing something error-prone, and its output format (frequencies, times, power)
plugs directly into a heatmap plot.


In [ ]:
def average_spectrogram_db(snippet_list, baseline_power_list, fs, nperseg=SPEC_NPERSEG):
    """
    Average spectrogram across trials and channels, expressed as dB relative to each trial's own
    pre-Poke-In baseline (see compute_trial_baseline_power in Part B).

    Baseline-correcting PER TRIAL before averaging (rather than baseline-correcting the final
    across-trial average) is the standard event-related-spectral-perturbation approach: it also
    cancels out session-to-session and channel-to-channel differences in absolute power before
    trials get combined, instead of letting the loudest trials/channels dominate the average.
    """
    all_Sxx_db = []
    freqs, times = None, None
    for snip, baseline_power in zip(snippet_list, baseline_power_list):
        n_channels = snip.shape[0]
        channel_Sxx = []
        for ch in range(n_channels):
            f, t, Sxx = signal.spectrogram(snip[ch], fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
            channel_Sxx.append(Sxx)
        trial_Sxx = np.mean(channel_Sxx, axis=0)
        freqs, times = f, t
        trial_Sxx_db = 10 * np.log10(trial_Sxx / baseline_power[:, None])
        all_Sxx_db.append(trial_Sxx_db)
    return freqs, times, np.mean(all_Sxx_db, axis=0)


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col, session_name in enumerate(snippets_by_rat):
    data = snippets_by_rat[session_name]
    inseq_snips = [s for s, lab in zip(data['snippets'], data['inseq_labels']) if lab == 1]
    inseq_baseline = [b for b, lab in zip(data['baseline_power'], data['inseq_labels']) if lab == 1]
    outseq_snips = [s for s, lab in zip(data['snippets'], data['inseq_labels']) if lab == 0]
    outseq_baseline = [b for b, lab in zip(data['baseline_power'], data['inseq_labels']) if lab == 0]
    fs = data['fs']

    groups = [
        ('InSeq', inseq_snips, inseq_baseline),
        ('OutSeq', outseq_snips, outseq_baseline),
    ]
    for row, (label, snips, baselines) in enumerate(groups):
        freqs, times, Sxx_db_avg = average_spectrogram_db(snips, baselines, fs)
        times_relative_ms = (times - SNIPPET_PRE_MS / 1000) * 1000  # 0 = Poke-In

        ax = axes[row, col]
        vmax = np.nanmax(np.abs(Sxx_db_avg))
        im = ax.pcolormesh(times_relative_ms, freqs, Sxx_db_avg, shading='auto', cmap='RdBu_r',
                            vmin=-vmax, vmax=vmax)
        ax.axvline(0, color='black', linestyle='--', linewidth=1, label='Poke-In')
        ax.set_ylim(0, 150)
        ax.set_title(f"{session_name.split('_')[1]} - {label} (n={len(snips)}), Poke-In aligned")
        ax.set_xlabel('Time relative to Poke-In (ms)')
        ax.set_ylabel('Frequency (Hz)')
        if row == 0 and col == 0:
            ax.legend(fontsize=8)
        plt.colorbar(im, ax=ax, label='Power vs. pre-Poke-In baseline (dB)')

plt.tight_layout()
plt.show()


## Part D: Same Spectrograms, Poke-Out Aligned

Same data, re-centered on each trial's own Poke-Out moment (from Part A) instead of Poke-In, so
whatever happens right as the rat's nose LEAVES the port is visible directly, rather than inferred.


In [ ]:
def spectrogram_db_from_recentered(re_centered, baseline_power, fs, nperseg=SPEC_NPERSEG):
    """
    Spectrogram of an already Poke-Out-re-centered snippet, normalized against the SAME
    pre-Poke-In baseline used in Part C (passed in via baseline_power). This re-centered window
    starts well after Poke-In, so it no longer contains that baseline period itself -- we have to
    carry the baseline over from Part A/B rather than recompute it here.
    """
    n_channels = re_centered.shape[0]
    channel_Sxx = []
    for ch in range(n_channels):
        f, t, Sxx = signal.spectrogram(re_centered[ch], fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
        channel_Sxx.append(Sxx)
    trial_Sxx = np.mean(channel_Sxx, axis=0)
    trial_Sxx_db = 10 * np.log10(trial_Sxx / baseline_power[:, None])
    return f, t, trial_Sxx_db


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col, session_name in enumerate(snippets_by_rat):
    data = snippets_by_rat[session_name]
    fs = data['fs']

    inseq_entries, outseq_entries = [], []
    for snip, lab, gap_ms, baseline_power in zip(
        data['snippets'], data['inseq_labels'], data['poke_out_gaps_ms'], data['baseline_power']
    ):
        if np.isnan(gap_ms):
            continue
        # snippet's sample 0 corresponds to (Poke-In - 1000ms); Poke-Out sample index within snippet:
        pokeout_sample_in_snip = int(round((gap_ms / 1000 + SNIPPET_PRE_MS / 1000) * fs))
        pre_samples = int(round(1.0 * fs))   # 1 second before Poke-Out
        post_samples = int(round(2.0 * fs))  # 2 seconds after Poke-Out
        start = pokeout_sample_in_snip - pre_samples
        end = pokeout_sample_in_snip + post_samples
        if start < 0 or end > snip.shape[1]:
            continue
        re_centered = snip[:, start:end]
        entry = (re_centered, baseline_power)
        if lab == 1:
            inseq_entries.append(entry)
        else:
            outseq_entries.append(entry)

    for row, (label, entries) in enumerate([('InSeq', inseq_entries), ('OutSeq', outseq_entries)]):
        if len(entries) == 0:
            continue
        all_Sxx_db, freqs, times = [], None, None
        for re_centered, baseline_power in entries:
            freqs, times, trial_Sxx_db = spectrogram_db_from_recentered(re_centered, baseline_power, fs)
            all_Sxx_db.append(trial_Sxx_db)
        Sxx_db_avg = np.mean(all_Sxx_db, axis=0)
        half_window_ms = (SPEC_NPERSEG / (2 * fs)) * 1000
        times_relative_ms = (times - 1.0) * 1000 + half_window_ms

        ax = axes[row, col]
        vmax = np.nanmax(np.abs(Sxx_db_avg))
        im = ax.pcolormesh(times_relative_ms, freqs, Sxx_db_avg, shading='auto', cmap='RdBu_r',
                            vmin=-vmax, vmax=vmax)
        ax.axvline(0, color='black', linestyle='--', linewidth=1, label='Poke-Out')
        ax.set_ylim(0, 150)
        ax.set_title(f"{session_name.split('_')[1]} - {label} (n={len(entries)}), Poke-Out aligned")
        ax.set_xlabel('Time relative to Poke-Out (ms)')
        ax.set_ylabel('Frequency (Hz)')
        if row == 0 and col == 0:
            ax.legend(fontsize=8)
        plt.colorbar(im, ax=ax, label='Power vs. pre-Poke-In baseline (dB)')

plt.tight_layout()
plt.show()


## Part E: What to Look For (Guidance for Interpreting the Figures Above)

- **Horizontal bands of color** = sustained power at a specific frequency over time, e.g. a persistent
  bright band around 4-12 Hz would indicate theta oscillation is present throughout, not band power
  summed into one number.
- **Differences between the InSeq and OutSeq panels** (same rat, same alignment) point to which
  frequencies and times actually distinguish the two conditions, this should directly inform which bands
  to prioritize in the GLM, rather than the delta/theta/beta/gamma split chosen up front by convention.
- **Differences between Poke-In-aligned and Poke-Out-aligned panels** tell us whether the informative
  signal is better described relative to nose-in or nose-out, directly relevant to the "should we center
  on Poke-Out instead" question raised by the lab.
- **Whether to collapse time or not**: if a frequency band's power is fairly constant over the whole
  window, collapsing time (summing power into one number, what we've done throughout this project) loses
  little. If a band's power changes sharply at a specific moment, collapsing time would blur that
  moment away, and a time-resolved feature (or a carefully chosen short window right at that moment)
  would preserve more signal.


## Part F: Text-Only Results Export


In [ ]:
import json as _json
import os

export_summary = {
    "purpose": "Corrected Poke-In/Poke-Out extraction (bounded per-trial), spectrogram EDA groundwork",
    "poke_in_to_poke_out_gap_ms": {
        session_name: {
            "median": float(np.nanmedian(corrected_results[session_name]['poke_out_gap_ms'])),
            "min": float(np.nanmin(corrected_results[session_name]['poke_out_gap_ms'])),
            "max": float(np.nanmax(corrected_results[session_name]['poke_out_gap_ms'])),
            "std": float(np.nanstd(corrected_results[session_name]['poke_out_gap_ms'])),
            "n_found": int((~np.isnan(corrected_results[session_name]['poke_out_gap_ms'])).sum()),
            "n_total": len(corrected_results[session_name]['trial_idx']),
        }
        for session_name in corrected_results
    },
}

print(_json.dumps(export_summary, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/notebook020_pokeout_spectrogram_results.json', 'w') as f:
    _json.dump(export_summary, f, indent=2)
print("\nSaved to outputs/logs/notebook020_pokeout_spectrogram_results.json")


## Part G: Written Summary Report

**Objective**

Fix the cross-trial contamination bug found in notebook 019's Poke-Out extraction, and produce the
InSeq-vs-OutSeq spectrogram EDA requested directly by the lab, to let the actual frequency-vs-time data
guide band and window decisions rather than continuing with the convention-based band split used so far.

**Method**

Poke-Out was re-extracted per trial, with the search for the second `PokeEvents` entry strictly bounded
between the current trial's Poke-In and the next trial's Poke-In, eliminating the possibility of
capturing a future trial's event. Time-frequency spectrograms were computed per trial per channel
(`scipy.signal.spectrogram`), then averaged across channels and across trials within each condition
(InSeq, OutSeq), for both Poke-In-aligned and Poke-Out-aligned views, for Mitt and Buchanan.

**Results**

*(Fill in after running Parts A, C, D.)* Is the corrected Poke-In to Poke-Out gap now plausible and
consistent (no more multi-minute outliers)? Looking at the spectrograms: which frequency bands show
visibly different power between InSeq and OutSeq? Does anything change meaningfully around Poke-Out that
isn't visible in the Poke-In-aligned view?

**Interpretation**

*(Fill in after review.)* This section should directly answer: should band selection change based on
what's visible here? Should time be collapsed (summed into one number per band, as done throughout this
project) or preserved (a time-resolved feature)? Should the universal window be defined relative to
Poke-In or Poke-Out?

**Next Steps**

1. Use these figures to select a small, justified set of bands (rather than the convention-based
   delta/theta/beta/low-gamma/high-gamma split), directly citing what's visible here.
2. Settle on ONE universal window (length and reference point) applied identically to all 5 rats, chosen
   to work well for Mitt and Buchanan specifically.
3. Extend this same spectrogram EDA to the remaining 3 rats (Barat, Stella, Superchris) to confirm the
   pattern generalizes before finalizing the universal window.
4. Add AUC scoring alongside balanced accuracy in the next round of final models, per the lab's
   validation feedback (permutation testing is already in place from notebooks 014/016).
